# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library and referencing all elements by their Croissant `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset's Croissant JSON-LD file URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the mlcroissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Show summary metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\n" + f"Identifier: {metadata.identifier}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant dataset organizes its tabular data into *record sets*, each with a unique `@id`. Each record set defines *fields* (`@id`), which may correspond to columns in the data files. We'll list all record set and field `@id`s from the metadata.

In [ ]:
# List all record sets with their @id and display their fields and field @ids
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s).")
all_record_set_ids = []
for rs in record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name if hasattr(field, 'name') else 'N/A'})")
    all_record_set_ids.append(rs.id)
# For EDA and extraction, use the first record set as example
if all_record_set_ids:
    main_record_set_id = all_record_set_ids[0]
    print(f"\nExample RecordSet selected for extraction: {main_record_set_id}")

## 3. Data Extraction
Load data from the record set(s) into Pandas DataFrame(s) for analysis. 

We use the record set and field `@id`s from the overview above. Data is loaded and accessible for analytic operations.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet '{record_set_id}' with columns:")
    print(list(df.columns))

# Preview the first 5 rows for the main record set
if main_record_set_id in dataframes:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps using the dataset. We will:
- Select and filter over a *numeric* field using its `@id`
- Normalize the numeric field
- Optionally group by a *categorical* field `@id` if present

This example assumes at least one numeric field exists. Replace `<numeric_field_id>` and `<group_field_id>` with actual field `@id`s as revealed above.

In [ ]:
# Identify candidate numeric and group fields for analysis:
df = dataframes[main_record_set_id]
# Print a sample row to infer field types
print("Sample row:\n", df.iloc[0])
print("\nColumns:", list(df.columns))

# Attempt to automatically guess a numeric field (e.g., age or interval)
numeric_column_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi' and col != '@id']
if numeric_column_candidates:
    numeric_field_id = numeric_column_candidates[0]
    print(f"Selected numeric field: {numeric_field_id}")
else:
    # Try to pick a common name if auto-inference fails
    possible_numeric = [c for c in df.columns if 'age' in c.lower() or 'interval' in c.lower()]
    numeric_field_id = possible_numeric[0] if possible_numeric else None
    print(f"Fallback numeric field: {numeric_field_id}")

# Filter on numeric field
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10  # Use mean as threshold if numeric
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df = filtered_df.copy()  # Avoid SettingWithCopyWarning
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally group by a field (e.g., sex, status, or a categorical variable)
    categorical_fields = [col for col in df.columns if df[col].dtype == object and '@' not in col and col != numeric_field_id]
    group_field_id = None
    if categorical_fields:
        group_field_id = categorical_fields[0]
        print(f"\nGrouping by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical grouping field found.")
else:
    print("No numeric field could be identified for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and/or its relationship with the grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 6))
    sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group, if group_field_id exists
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(8, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion

- We have loaded and explored the FAIR^2 dataset using `mlcroissant`, referencing all entities by their Croissant `@id`s.
- We inspected available record sets and fields, and loaded records into pandas DataFrames.
- Basic data filtering, normalization, aggregation, and simple visualizations were demonstrated.
- For advanced analyses, please refer to the dataset's documentation and the full Croissant metadata to identify all available fields and their usage.

**Note:** Always reference fields by their `@id` for maximum reproducibility across code and metadata evolution.

For more complex operations (e.g., cross-record-set joins, advanced statistical testing), extend this notebook as needed and consult the [mlcroissant documentation](https://github.com/mlcommons/croissant) for additional API usage.